# Task 1

In [ ]:
from collections import Counter
import numpy as N

# Corpus definition
corpus = ["low", "new", "lowest", "newest", "widest", "lower"]
END_OF_WORD = "</w>"


def split_words(words):
    """Represent every word as characters followed by an end marker."""
    return [list(word) + [END_OF_WORD] for word in words]


def count_pairs(words):
    """Count adjacent symbol pairs without crossing word boundaries."""
    pair_counts = Counter()

    for symbols in words:
        for index in range(len(symbols) - 1):
            pair = (symbols[index], symbols[index + 1])
            pair_counts[pair] += 1

    return pair_counts


def merge_pair(words, pair):
    """Merge every non-overlapping occurrence of pair in every word."""
    merged_words = []

    for symbols in words:
        merged = []
        index = 0

        while index < len(symbols):
            if (
                index < len(symbols) - 1
                and symbols[index] == pair[0]
                and symbols[index + 1] == pair[1]
            ):
                merged.append(pair[0] + pair[1])
                index += 2
            else:
                merged.append(symbols[index])
                index += 1

        merged_words.append(merged)

    return merged_words


def train_bpe(words, number_of_merges):
    """Learn BPE merges, choosing ties by first occurrence in the corpus."""
    learned_merges = []
    current_words = [symbols.copy() for symbols in words]

    for merge_number in range(1, number_of_merges + 1):
        pair_counts = count_pairs(current_words)

        if not pair_counts:
            break

        # Counter.most_common() keeps the first-seen pair when counts tie.
        pair, frequency = pair_counts.most_common(1)[0]
        learned_merges.append(pair)
        current_words = merge_pair(current_words, pair)

        print(f"\nMerge {merge_number}: {pair} -> {pair[0] + pair[1]}")
        print(f"Frequency: {frequency}")
        print("Updated corpus:")
        for symbols in current_words:
            print(" ".join(symbols))

    return learned_merges, current_words


def apply_merges(word, learned_merges):
    """Apply learned BPE merges to one previously unseen word."""
    tokenized_word = split_words([word])
    tokenized_word = tokenized_word[0]

    for pair in learned_merges:
        tokenized_word = merge_pair([tokenized_word], pair)[0]

    return tokenized_word


# Start from word forms split into characters.
word_symbols = split_words(corpus)

# Display the initial vocabulary with NumPy.
list_inp = [symbol for word in word_symbols for symbol in word]
res = N.array(list_inp)
unique_res = N.unique(res)

print("Initial corpus:")
for word in word_symbols:
    print(" ".join(word))

print("\nUnique initial symbols using numpy.unique():")
print(unique_res)

# Count pairs, merge the most frequent pair, and repeat for five operations.
learned_merges, final_corpus = train_bpe(word_symbols, number_of_merges=5)

print("\nLearned merges in order:")
for merge_number, pair in enumerate(learned_merges, start=1):
    print(f"{merge_number}. {pair} -> {pair[0] + pair[1]}")


# Apply the learned merges to an unseen word in the same order.
test_word = "slowest"
test_tokens = apply_merges(test_word, learned_merges)

print(f"\nTokenization of unseen word '{test_word}':")
print(" ".join(test_tokens))

Initial corpus:
l o w </w>
n e w </w>
l o w e s t </w>
n e w e s t </w>
w i d e s t </w>
l o w e r </w>

Unique initial symbols using numpy.unique():
['</w>' 'd' 'e' 'i' 'l' 'n' 'o' 'r' 's' 't' 'w']

Merge 1: ('l', 'o') -> lo
Frequency: 3
Updated corpus:
lo w </w>
n e w </w>
lo w e s t </w>
n e w e s t </w>
w i d e s t </w>
lo w e r </w>

Merge 2: ('lo', 'w') -> low
Frequency: 3
Updated corpus:
low </w>
n e w </w>
low e s t </w>
n e w e s t </w>
w i d e s t </w>
low e r </w>

Merge 3: ('e', 's') -> es
Frequency: 3
Updated corpus:
low </w>
n e w </w>
low es t </w>
n e w es t </w>
w i d es t </w>
low e r </w>

Merge 4: ('es', 't') -> est
Frequency: 3
Updated corpus:
low </w>
n e w </w>
low est </w>
n e w est </w>
w i d est </w>
low e r </w>

Merge 5: ('est', '</w>') -> est</w>
Frequency: 3
Updated corpus:
low </w>
n e w </w>
low est</w>
n e w est</w>
w i d est</w>
low e r </w>

Learned merges in order:
1. ('l', 'o') -> lo
2. ('lo', 'w') -> low
3. ('e', 's') -> es
4. ('es', 't') -> est
5.

# Task 4

In [ ]:
! pip install torch


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
! pip install transformers


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

C:\Users\User\Desktop\DM\github\DL4NLP_2026\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\Desktop\DM\github\DL4NLP_2026\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate develope

In [ ]:
prompt = "The weather today is very"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
print("Prompt:", prompt)
print("Token ids:", input_ids[0].tolist())
print("Tokens:", tokenizer.convert_ids_to_tokens(input_ids[0]))

Prompt: The weather today is very
Token ids: [464, 6193, 1909, 318, 845]
Tokens: ['The', 'Ġweather', 'Ġtoday', 'Ġis', 'Ġvery']


In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=2,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,  # GPT-2 has no pad token
)

In [ ]:
new_ids = outputs[0, input_ids.shape[1]:]
print("\nGenerated token ids:", new_ids.tolist())
print("Generated tokens:", tokenizer.convert_ids_to_tokens(new_ids))
print("Continuation:", repr(tokenizer.decode(new_ids)))
print("Full text:", tokenizer.decode(outputs[0]))


Generated token ids: [922, 13]
Generated tokens: ['Ġgood', '.']
Continuation: ' good.'
Full text: The weather today is very good.
